# 03. Checking pySTAMPS outputs against STAMPS

This notebook explains how pySTAMPS checks whether one run folder matches a trusted STAMPS folder.

Verification matters because a processing run can finish without crashing and still produce the wrong result.


## What the STAMPS comparison means

The STAMPS folder is the trusted baseline in this notebook.

In plain language, verification asks:

- did pySTAMPS create the files you expected?
- do the important numeric values match within tolerance?
- if they do not match, which file should you inspect first?


In [1]:
from pathlib import Path

DATASET_MAIN = Path('inputs_and_outputs/InSAR_dataset_test')
DATASET_STAGE8 = Path('inputs_and_outputs/InSAR_dataset_test_stage8diag')

for path in (DATASET_MAIN, DATASET_STAGE8):
    print(f'{path}:', 'present' if path.exists() else 'missing')


inputs_and_outputs/InSAR_dataset_test: missing
inputs_and_outputs/InSAR_dataset_test_stage8diag: missing


## The generic verification command

This is the command shape you should learn first because it works with arbitrary paths.


In [2]:
generic_verify = 'uv run pystamps verify --run /path/to/run_dataset --golden /path/to/golden_dataset'
print(generic_verify)


uv run pystamps verify --run /path/to/run_dataset --golden /path/to/golden_dataset


## Repository-specific example

This repository includes bundled STAMPS dataset paths that are useful for examples. This cell only prints the command.


In [3]:
repo_verify = (
    'uv run pystamps verify '
    '--run inputs_and_outputs/InSAR_dataset_test '
    '--golden inputs_and_outputs/InSAR_dataset_test'
)
print(repo_verify)


uv run pystamps verify --run inputs_and_outputs/InSAR_dataset_test --golden inputs_and_outputs/InSAR_dataset_test


## Important distinction: generic verify versus `make verify`

Use generic `pystamps verify --run ... --golden ...` when you want to compare arbitrary paths.

Use `make verify` only when you intentionally want the repository's fixed convenience case. It is not a general substitute for the CLI command above.


In [4]:
print('make verify  # fixed-path convenience target for this repository only')


make verify  # fixed-path convenience target for this repository only


## How to interpret a verification result

A verification summary usually answers three questions:

1. Did the comparison succeed overall?
2. How many files or checks were examined?
3. If there were failures, which file paths and messages should you inspect first?

That means verification is not only a pass/fail gate. It is also a debugging tool.


## Release-grade audit execution

This section runs the repository's strict parity audit with an explicit Rust/native stage-2 config.

The notebook config uses:

- `runtime.stage2_kernel_backend: native`
- `runtime.stage2_native_threads: 0`

In pySTAMPS, `stage2_native_threads: 0` means *use the full detected CPU budget for stage 2*.

In [5]:
import json
import os
import subprocess
import time
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'pystamps').exists():
    REPO_ROOT = REPO_ROOT.parent
AUDIT_CONFIG = REPO_ROOT / 'notebooks' / '03_pystamps_verification.audit.yaml'
AUDIT_OUTPUT = REPO_ROOT / 'inputs_and_outputs' / 'validation_runs' / 'latest_audit_rust_notebook.json'
AUDIT_STDOUT = REPO_ROOT / 'inputs_and_outputs' / 'validation_runs' / 'latest_audit_rust_notebook.stdout.txt'
VERIFY_STDOUT = REPO_ROOT / 'inputs_and_outputs' / 'validation_runs' / 'latest_verify_rust_notebook.txt'
SUMMARY_OUTPUT = REPO_ROOT / 'inputs_and_outputs' / 'validation_runs' / 'latest_audit_rust_notebook_summary.json'
AUDIT_DATASETS = [
    REPO_ROOT / 'inputs_and_outputs' / 'InSAR_dataset_test_stage8diag',
    REPO_ROOT / 'inputs_and_outputs' / 'InSAR_dataset_test',
]
AUDIT_ENV = os.environ.copy()
AUDIT_ENV.update({
    'OPENBLAS_NUM_THREADS': '1',
    'OMP_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1',
    'PYTHONPATH': str(REPO_ROOT),
})
AUDIT_CMD = [
    'uv', 'run', 'python', 'scripts/validate_audit.py',
    '--config', str(AUDIT_CONFIG),
    '--datasets', *(str(path) for path in AUDIT_DATASETS),
    '--output', str(AUDIT_OUTPUT),
]

print(json.dumps({
    'audit_config': str(AUDIT_CONFIG),
    'stage2_kernel_backend': 'native',
    'stage2_native_threads': 0,
    'cpu_count': os.cpu_count(),
    'audit_output': str(AUDIT_OUTPUT),
    'audit_command': AUDIT_CMD,
}, indent=2))

{
  "audit_config": "/shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/notebooks/03_pystamps_verification.audit.yaml",
  "stage2_kernel_backend": "native",
  "stage2_native_threads": 0,
  "cpu_count": 16,
  "audit_output": "/shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/inputs_and_outputs/validation_runs/latest_audit_rust_notebook.json",
  "audit_command": [
    "uv",
    "run",
    "python",
    "scripts/validate_audit.py",
    "--config",
    "/shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/notebooks/03_pystamps_verification.audit.yaml",
    "--datasets",
    "/shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/inputs_and_outputs/InSAR_dataset_test_stage8diag",
    "/shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/inputs_and_outputs/InSAR_dataset_test",
    "--output",
    "/shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/inputs_and_outputs/validation_runs/latest_audit_rust_notebook.json"
  ]
}


In [6]:
if not all(path.exists() for path in AUDIT_DATASETS):
    print('Required local datasets are missing; audit execution skipped.')
    audit_payload = None
    audit_elapsed_sec = None
else:
    started = time.perf_counter()
    audit_proc = subprocess.run(AUDIT_CMD, cwd=REPO_ROOT, env=AUDIT_ENV, capture_output=True, text=True)
    audit_elapsed_sec = time.perf_counter() - started
    AUDIT_STDOUT.write_text(audit_proc.stdout, encoding='utf-8')
    if audit_proc.stderr:
        AUDIT_STDOUT.with_suffix('.stderr.txt').write_text(audit_proc.stderr, encoding='utf-8')
    if audit_proc.returncode not in (0, 1):
        raise RuntimeError(audit_proc.stderr[-4000:])
    audit_payload = json.loads(AUDIT_OUTPUT.read_text(encoding='utf-8'))
    audit_rows = []
    for audit in audit_payload.get('audits', []):
        generation = audit.get('run_generation', {})
        audit_rows.append({
            'dataset': audit['dataset'],
            'status': audit['status'],
            'checked': audit['checked'],
            'failed': audit['failed'],
            'run_root': audit['run_root'],
            'start_step': generation.get('start_step'),
            'end_step': generation.get('end_step'),
            'seed_name': generation.get('seed_name'),
        })
    print(json.dumps({
        'audit_returncode': audit_proc.returncode,
        'audit_elapsed_sec': round(audit_elapsed_sec, 2),
        'audit_ok': audit_payload.get('ok'),
        'failed_workflows': audit_payload.get('failed_workflows'),
        'audits': audit_rows,
    }, indent=2))

{
  "audit_returncode": 1,
  "audit_elapsed_sec": 1932.03,
  "audit_ok": false,
  "failed_workflows": [
    "full_validation"
  ],
  "audits": [
    {
      "dataset": "InSAR_dataset_test_stage8diag",
      "status": "failed",
      "checked": 14,
      "failed": 12,
      "run_root": "/shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/inputs_and_outputs/validation_runs/20260330_162544/InSAR_dataset_test_stage8diag_stage2_8",
      "start_step": 2,
      "end_step": 8,
      "seed_name": "InSAR_dataset_test_stage8diag"
    },
    {
      "dataset": "InSAR_dataset_test",
      "status": "failed",
      "checked": 14,
      "failed": 13,
      "run_root": "/shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/inputs_and_outputs/validation_runs/20260330_162544/InSAR_dataset_test_stage2_8",
      "start_step": 2,
      "end_step": 8,
      "seed_name": "RUN_FULL_GATE_1e10"
    }
  ]
}


## Explicit verify using the fresh full-dataset run root

For release-grade parity work, the explicit verify step must use the `run_root` recorded in the fresh audit artifact.

In [7]:
verify_summary = None
if audit_payload and audit_payload.get('audits'):
    full_audit = next((audit for audit in audit_payload['audits'] if audit['dataset'] == 'InSAR_dataset_test'), None)
    if full_audit is None:
        print('Full dataset audit row was not present; explicit verify skipped.')
    else:
        verify_cmd = [
            'uv', 'run', 'pystamps', 'verify',
            '--run', str(full_audit['run_root']),
            '--golden', str(REPO_ROOT / 'inputs_and_outputs' / 'InSAR_dataset_test'),
        ]
        started = time.perf_counter()
        verify_proc = subprocess.run(verify_cmd, cwd=REPO_ROOT, env=AUDIT_ENV, capture_output=True, text=True)
        verify_elapsed_sec = time.perf_counter() - started
        VERIFY_STDOUT.write_text(verify_proc.stdout, encoding='utf-8')
        if verify_proc.stderr:
            VERIFY_STDOUT.with_suffix('.stderr.txt').write_text(verify_proc.stderr, encoding='utf-8')
        if verify_proc.returncode not in (0, 1):
            raise RuntimeError(verify_proc.stderr[-4000:])
        verify_summary = {
            'verify_returncode': verify_proc.returncode,
            'verify_elapsed_sec': round(verify_elapsed_sec, 2),
            'verify_stdout_path': str(VERIFY_STDOUT),
        }
        print(verify_proc.stdout)
        summary_payload = {
            'stage2_kernel_backend': 'native',
            'stage2_native_threads': 0,
            'audit_elapsed_sec': round(audit_elapsed_sec, 2),
            'verify_elapsed_sec': round(verify_elapsed_sec, 2),
            'total_elapsed_sec': round(audit_elapsed_sec + verify_elapsed_sec, 2),
            'audit_ok': audit_payload.get('ok'),
            'failed_workflows': audit_payload.get('failed_workflows'),
            'full_run_root': full_audit['run_root'],
            'audit_output': str(AUDIT_OUTPUT),
            'audit_stdout_path': str(AUDIT_STDOUT),
            **verify_summary,
        }
        SUMMARY_OUTPUT.write_text(json.dumps(summary_payload, indent=2), encoding='utf-8')
        print(json.dumps(summary_payload, indent=2))
else:
    print('Audit payload was not available; explicit verify skipped.')

{
  "ok": false,
  "checked": 14,
  "failed": [
    {
      "path": "PATCH_1/pm1.mat",
      "message": "Value mismatch for key 'C_ps', max_abs=1.93541"
    },
    {
      "path": "PATCH_1/select1.mat",
      "message": "Value mismatch for key 'C_ps2', max_abs=0.0372478"
    },
    {
      "path": "PATCH_1/weed1.mat",
      "message": "Value mismatch for key 'ps_max', max_abs=0.0101583"
    },
    {
      "path": "ps2.mat",
      "message": "Value mismatch for key 'bperp', max_abs=6.4859"
    },
    {
      "path": "pm2.mat",
      "message": "Shape mismatch for key 'C_ps': (77888,) != (587320,)"
    },
    {
      "path": "ph2.mat",
      "message": "Shape mismatch for key 'ph': (77888, 76) != (587320, 76)"
    },
    {
      "path": "phuw2.mat",
      "message": "Value mismatch for key 'msd', max_abs=0.894706"
    },
    {
      "path": "scla2.mat",
      "message": "Shape mismatch for key 'C_ps_uw': (77888,) != (587320,)"
    },
    {
      "path": "mean_v.mat",
      "message": "Sh

## Next step

After working through this notebook set, you should be comfortable with the pySTAMPS exploration pattern:

1. inspect the dataset
2. preview execution with `--dry-run`
3. verify outputs with the generic CLI command


## Derived step timing table

In [8]:
from pathlib import Path
import json
import subprocess

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'pystamps').exists():
    REPO_ROOT = REPO_ROOT.parent
TIMING_OUTPUT = REPO_ROOT / 'inputs_and_outputs' / 'validation_runs' / 'latest_audit_rust_notebook_stage_timings.json'
if not TIMING_OUTPUT.exists():
    subprocess.run(
        [
            'uv',
            'run',
            'python',
            'scripts/derive_audit_stage_timings.py',
            '--audit',
            str(REPO_ROOT / 'inputs_and_outputs' / 'validation_runs' / 'latest_audit_rust_notebook.json'),
            '--output',
            str(TIMING_OUTPUT),
        ],
        check=True,
    )

payload = json.loads(TIMING_OUTPUT.read_text(encoding='utf-8'))
print(json.dumps({'timing_method': payload['timing_method'], 'timing_output': str(TIMING_OUTPUT), 'notes': payload['notes']}, indent=2))
for audit in payload['audits']:
    print()
    print(f"{audit['dataset']} steps {audit['start_step']}-{audit['end_step']}")
    print('| Stage | Duration (s) | Start (UTC) | End (UTC) | Basis |')
    print('| --- | ---: | --- | --- | --- |')
    for row in audit['timings']:
        print(f"| {row['stage']} | {row['duration_sec']:.2f} | {row['start_utc']} | {row['end_utc']} | {row['basis']} |")


{
  "timing_method": "artifact_mtime_derived",
  "timing_output": "/shared/home/rdelprete/PythonProjects/AgenticWork/pySTAMPS/inputs_and_outputs/validation_runs/latest_audit_rust_notebook_stage_timings.json",
  "notes": [
    "Stage 2 duration uses the validation-run directory timestamp as the workflow start when stage 2 is regenerated.",
    "For workflows that start at stage 4 from a seeded run copy, stage 4 duration is derived from the span of freshly rewritten stage-4 artifacts because copied seed files preserve historical mtimes."
  ]
}

InSAR_dataset_test_stage8diag steps 2-8
| Stage | Duration (s) | Start (UTC) | End (UTC) | Basis |
| --- | ---: | --- | --- | --- |
| 2 | 740.37 | 2026-03-29T15:40:35+00:00 | 2026-03-29T15:52:55.372095+00:00 | validation_run_dir_timestamp |
| 3 | 623.76 | 2026-03-29T15:52:55.372095+00:00 | 2026-03-29T16:03:19.134806+00:00 | previous_stage_end |
| 4 | 43.00 | 2026-03-29T16:03:19.134806+00:00 | 2026-03-29T16:04:02.139245+00:00 | previous_stage_end |